# Criterios de selección de K y perfil de los 8 clústeres (Supplementary Figures S1 y S10)

Notebook independiente que reproduce las dos figuras suplementarias que **no** vienen de ninguno de los 9 notebooks de reanalisis, porque pertenecen al analisis principal original (no a una respuesta puntual a un revisor):

- **Figura S1**: criterios de selección de K (codo/inercia, Silhouette, Davies-Bouldin) para K=2 a K=8, sobre la muestra de Fase 1 (n=200,000, semilla 42) — la misma que ya usaste para elegir K=8.
- **Figura S10**: perfil estandarizado (z-score) de los 8 clústeres de la partición publicada, sobre variables socioeconómicas, de financiamiento, acceso digital y desempeño.

**Cómo correrlo:**

1. Entorno de ejecución → Cambiar tipo de entorno → CPU (no necesita GPU).
2. Ajusta la ruta de tu `df_maestra.csv` en la celda de carga de datos si no está en `/content/drive/MyDrive/Proyecto/`.
3. Corre todas las celdas en orden (**Entorno de ejecución → Ejecutar todas**).
4. Los resultados se guardan automáticamente en tu Drive, en `/content/drive/MyDrive/Proyecto/seleccion_k_y_perfiles/`.

⏱️ La Figura S1 (K=2 a 8 sobre 200,000 filas) es la parte más pesada — unos 15-25 minutos. La Figura S10 reutiliza la misma partición K=8 que ya usan los demás notebooks y tarda unos 5-10 minutos adicionales.

In [ ]:
!pip install -q umap-learn

In [ ]:
import os, time, json
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.cluster import MiniBatchKMeans, KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
import umap.umap_ as umap_cpu
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
print("\u2705 Librer\u00edas listas")

## 1. Cargar los datos desde Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv('/content/drive/MyDrive/Proyecto/df_maestra.csv')
df = df.loc[:, ~df.columns.str.contains('^Unnamed|^Column1')]
OUT_DIR = '/content/drive/MyDrive/Proyecto/seleccion_k_y_perfiles'
os.makedirs(OUT_DIR, exist_ok=True)
print(f"Shape original: {df.shape}")
df.head(3)

## 2. Preprocesamiento (idéntico al notebook original)

In [ ]:
def preprocesar_saber_pro(df_raw, sample_n=200_000, random_state=42):
    df_limpio = df_raw.copy()

    mapa_bano = {'1': 1, '2': 2, '3 o 4': 3, '5 o 6': 5, 'MAS DE 6': 6, 'NINGUNA': 0}
    mapa_estrato = {'Sin estrato': 0, 'Estrato 1': 1, 'Estrato 2': 2,
                     'Estrato 3': 3, 'Estrato 4': 4, 'Estrato 5': 5, 'Estrato 6': 6}
    mapa_valormatricula = {
        'Sin costo': 0, 'Menos de 500 mil': 1,
        'Entre 500 mil y menos de 1 mill\u00f3n': 2,
        'Entre 1 mill\u00f3n y menos de 2.5 millones': 3,
        'Entre 2.5 millones y menos de 4 millones': 4,
        'Entre 4 millones y menos de 5.5 millones': 5,
        'Entre 5.5 millones y menos de 7 millones': 6,
        'M\u00e1s de 7 millones': 7}
    mapa_educ = {
        'Ninguno': 0, 'Primaria incompleta': 1, 'Primaria completa': 2,
        'Secundaria (Bachillerato) incompleta': 3,
        'Secundaria (Bachillerato) completa': 4,
        'T\u00e9cnica o tecnol\u00f3gica incompleta': 5,
        'T\u00e9cnica o tecnol\u00f3gica completa': 6,
        'Educaci\u00f3n profesional incompleta': 7,
        'EDUCACI\u00d3N PROFESIONAL COMPLETA': 8, 'POSTGRADO': 9}
    mapeo_horas = {'0': 0, 'Menos de 10 horas': 1, 'Entre 11 y 20 horas': 2,
                    'Entre 21 y 30 horas': 3, 'M\u00e1s de 30 horas': 4}
    mapeo_semestre = {str(i).zfill(2): i for i in range(1, 12)}
    mapeo_semestre['12 o m\u00e1s'] = 12

    mapeables = {
        'FAMI_CUANTOSCOMPARTEBA\u00d1O':      mapa_bano,
        'FAMI_ESTRATOVIVIENDA':          mapa_estrato,
        'ESTU_VALORMATRICULAUNIVERSIDAD': mapa_valormatricula,
        'FAMI_EDUCACIONPADRE':           mapa_educ,
        'FAMI_EDUCACIONMADRE':           mapa_educ,
        'ESTU_HORASSEMANATRABAJA':       mapeo_horas,
        'ESTU_SEMESTRECURSA':            mapeo_semestre,
    }
    for col, mapa in mapeables.items():
        if col in df_limpio.columns:
            df_limpio[col] = df_limpio[col].map(mapa)

    columnas_puntaje = [
        'MOD_RAZONA_CUANTITAT_PUNT', 'MOD_LECTURA_CRITICA_PUNT',
        'MOD_COMPETEN_CIUDADA_PUNT', 'MOD_INGLES_PUNT', 'MOD_COMUNI_ESCRITA_PUNT']
    columnas_ordinales = [
        'FAMI_ESTRATOVIVIENDA', 'ESTU_VALORMATRICULAUNIVERSIDAD',
        'FAMI_EDUCACIONPADRE', 'FAMI_EDUCACIONMADRE', 'ESTU_HORASSEMANATRABAJA']
    columnas_nominales = [
        'ESTU_TITULOOBTENIDOBACHILLER',
        'ESTU_PAGOMATRICULABECA', 'ESTU_PAGOMATRICULACREDITO',
        'ESTU_PAGOMATRICULAPADRES', 'ESTU_PAGOMATRICULAPROPIO',
        'ESTU_COMOCAPACITOEXAMENSB11',
        'FAMI_TIENEINTERNET', 'FAMI_TIENECOMPUTADOR',
        'FAMI_TIENEAUTOMOVIL', 'FAMI_TIENELAVADORA']
    col_geo = 'ESTU_COD_DEPTO_PRESENTACION'

    for col in columnas_puntaje:
        if col in df_limpio.columns:
            df_limpio[col] = pd.to_numeric(df_limpio[col], errors='coerce')
            df_limpio[col] = df_limpio[col].fillna(df_limpio[col].mean())
    for col in columnas_ordinales:
        if col in df_limpio.columns:
            df_limpio[col] = df_limpio[col].fillna(df_limpio[col].median())
    for col in columnas_nominales:
        if col in df_limpio.columns:
            df_limpio[col] = df_limpio[col].fillna(df_limpio[col].mode(dropna=True)[0])

    cols_usar = columnas_ordinales + columnas_puntaje + columnas_nominales
    cols_extra = [col_geo, 'INST_COD_INSTITUCION', 'ESTU_PRGM_ACADEMICO',
                  'PERIODO', 'PUNT_GLOBAL', 'ESTU_CONSECUTIVO']
    cols_df = cols_usar + [c for c in cols_extra if c in df_limpio.columns]
    df_filtrado = df_limpio[[c for c in cols_df if c in df_limpio.columns]].copy()
    df_filtrado = df_filtrado.dropna(subset=[c for c in columnas_puntaje if c in df_filtrado.columns])
    print(f"Filas despu\u00e9s de limpieza: {len(df_filtrado):,}")

    cols_punt = [c for c in columnas_puntaje if c in df_filtrado.columns]
    cols_ord = [c for c in columnas_ordinales if c in df_filtrado.columns]
    cols_nom = [c for c in columnas_nominales if c in df_filtrado.columns]

    scaler = StandardScaler()
    encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    X_punt = scaler.fit_transform(df_filtrado[cols_punt])
    X_ohe = encoder.fit_transform(df_filtrado[cols_nom])
    X = np.hstack([df_filtrado[cols_ord].values, X_punt, X_ohe])
    feature_names = (cols_ord + list(scaler.get_feature_names_out(cols_punt))
                      + list(encoder.get_feature_names_out(cols_nom)))

    if np.isnan(X).any():
        from sklearn.impute import SimpleImputer
        X = SimpleImputer(strategy='median').fit_transform(X)
        print("NaN residuales imputados con mediana")

    if sample_n is not None and sample_n < df_filtrado.shape[0]:
        rng = np.random.default_rng(seed=random_state)
        idx = rng.choice(df_filtrado.shape[0], size=sample_n, replace=False)
        df_filtrado = df_filtrado.iloc[idx].reset_index(drop=True)
        X = X[idx]

    print(f"Preprocesamiento completo \u2014 shape X: {X.shape}")
    return df_limpio, df_filtrado, X, feature_names, encoder, scaler


In [ ]:
df_limpio_full, df_filtrado_full, X_full, feature_names, encoder, scaler = \
    preprocesar_saber_pro(df, sample_n=None)
n_total = X_full.shape[0]
print(f"Shape X_full: {X_full.shape}")

## 3. Figura S1 — criterios de selección de K (Fase 1, n=200,000)
Reproduce la muestra de Fase 1 (semilla 42, n=200,000) y corre K-Means para K=2..8 sobre el embedding UMAP 2D, calculando inercia, Silhouette y Davies-Bouldin en cada K.

In [ ]:
SEED_FASE1 = 42
rng_f1 = np.random.default_rng(seed=SEED_FASE1)
idx_fase1 = rng_f1.choice(n_total, size=200_000, replace=False)
X_fase1 = X_full[idx_fase1]

print("Ajustando UMAP sobre la muestra de Fase 1 (n=200,000)...")
t0 = time.time()
reducer_f1 = umap_cpu.UMAP(n_components=2, random_state=SEED_FASE1, n_neighbors=10,
                            low_memory=True, n_jobs=-1)
if X_fase1.shape[0] > 100_000:
    idx_fit_f1 = rng_f1.choice(X_fase1.shape[0], size=80_000, replace=False)
    reducer_f1.fit(X_fase1[idx_fit_f1])
    emb_fase1 = reducer_f1.transform(X_fase1)
else:
    emb_fase1 = reducer_f1.fit_transform(X_fase1)
print(f"UMAP listo en {time.time()-t0:.1f}s. shape={emb_fase1.shape}")

In [ ]:
resultados_k = {"K": [], "inercia": [], "silhouette": [], "davies_bouldin": []}
for k in range(2, 9):
    t_k = time.time()
    km = KMeans(n_clusters=k, random_state=SEED_FASE1, n_init=10)
    labels_k = km.fit_predict(emb_fase1)
    sample = min(10_000, len(labels_k))
    sil = silhouette_score(emb_fase1, labels_k, sample_size=sample, random_state=42)
    db = davies_bouldin_score(emb_fase1, labels_k)
    resultados_k["K"].append(k)
    resultados_k["inercia"].append(float(km.inertia_))
    resultados_k["silhouette"].append(float(sil))
    resultados_k["davies_bouldin"].append(float(db))
    print(f"K={k}: inercia={km.inertia_:.0f} silhouette={sil:.4f} "
          f"davies_bouldin={db:.4f} ({time.time()-t_k:.1f}s)")

json.dump(resultados_k, open(os.path.join(OUT_DIR, 'resultados_seleccion_k.json'), 'w'), indent=2)
print("\nValores esperados (manuscrito, Fase 1): K=8 no es el \u00f3ptimo individual en "
      "ninguna m\u00e9trica (mejor Silhouette en K=2; mejor Davies-Bouldin en K=4), pero se "
      "mantiene cercano al mejor Davies-Bouldin.")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), dpi=150)
Ks = resultados_k["K"]

axes[0].plot(Ks, resultados_k["inercia"], marker='o', color='#3B6FA0')
axes[0].axvline(8, color='#B33F3F', linestyle='--', linewidth=1.2)
axes[0].set_xlabel('K'); axes[0].set_ylabel('Inercia'); axes[0].set_title('M\u00e9todo del codo (inercia)')

axes[1].plot(Ks, resultados_k["silhouette"], marker='o', color='#3B6FA0')
axes[1].axvline(8, color='#B33F3F', linestyle='--', linewidth=1.2, label='K=8 (elegido)')
axes[1].set_xlabel('K'); axes[1].set_ylabel('Silhouette'); axes[1].set_title('\u00cdndice de Silhouette')
axes[1].legend(fontsize=8)

axes[2].plot(Ks, resultados_k["davies_bouldin"], marker='o', color='#3B6FA0')
axes[2].axvline(8, color='#B33F3F', linestyle='--', linewidth=1.2)
axes[2].set_xlabel('K'); axes[2].set_ylabel('Davies-Bouldin (menor = mejor)')
axes[2].set_title('\u00cdndice Davies-Bouldin')

for ax in axes:
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.suptitle('Criterios de selecci\u00f3n de K \u2014 KMeans sobre UMAP 2D (Fase 1, n=200,000)')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'figura_seleccion_k.png'), dpi=200, bbox_inches='tight', facecolor='white')
plt.show()
print("\u2705 Figura S1 guardada en Drive")

## 4. Partición publicada de referencia (K=8, sobre las 452,020 filas)
Misma metodología que los demás notebooks: UMAP fit sobre 80,000 filas, transform sobre el resto, K-Means K=8.

In [ ]:
SEED_BASE = 42
K = 8
N_FIT = 80_000

rng_fit = np.random.default_rng(seed=SEED_BASE)
idx_fit = rng_fit.choice(n_total, size=N_FIT, replace=False)

print("Ajustando UMAP sobre 80,000 filas y transformando las 452,020...")
t0 = time.time()
reducer_base = umap_cpu.UMAP(n_components=2, random_state=SEED_BASE, n_neighbors=10,
                              low_memory=True, n_jobs=-1)
reducer_base.fit(X_full[idx_fit])
emb_full_base = reducer_base.transform(X_full)
km_base = MiniBatchKMeans(n_clusters=K, random_state=SEED_BASE, n_init="auto", batch_size=10_000)
labels_full = km_base.fit_predict(emb_full_base)
print(f"Listo en {time.time()-t0:.1f}s. Tama\u00f1os: {np.bincount(labels_full)}")

## 4.1. Realineacion de las etiquetas de cluster con la Tabla 7 publicada

Los numeros de cluster que produce K-Means son arbitrarios entre corridas independientes (ya lo advertiamos en la seccion 10 de este README para `effect_sizes_and_coverage.py`). Esta celda reordena las etiquetas de `labels_full` por rango de puntaje medio para que coincidan con la numeracion ya publicada en la Tabla 7 del manuscrito, en vez de dejar los numeros arbitrarios que salieron de esta corrida. Si vuelves a correr el notebook con datos o semillas distintas, esta celda se ajusta sola siempre que el ORDEN relativo de los 8 grupos por puntaje se mantenga; si dos clusters quedan con puntajes medios muy cercanos entre si, revisa el print de abajo antes de confiar en el reetiquetado.

In [ ]:
# Orden publicado en la Tabla 7 del manuscrito, de menor a mayor puntaje medio:
# Cluster 0 (135.92) < Cluster 6 (136.30) < Cluster 3 (140.92) < Cluster 2 (145.69)
# < Cluster 5 (148.48) < Cluster 1 (150.21) < Cluster 4 (161.98) < Cluster 7 (205.28)
ORDEN_TABLA7_ASCENDENTE = [0, 6, 3, 2, 5, 1, 4, 7]

puntaje_por_cluster_actual = (
    pd.Series(df_filtrado_full['PUNT_GLOBAL'].values)
    .groupby(labels_full).mean().sort_values()
)
orden_actual_ascendente = puntaje_por_cluster_actual.index.tolist()

mapa_reetiquetado = dict(zip(orden_actual_ascendente, ORDEN_TABLA7_ASCENDENTE))
print('Puntajes medios de esta corrida, de menor a mayor:')
print(puntaje_por_cluster_actual.round(2))
print('\nBrecha minima entre clusters consecutivos (puntos):',
      round(puntaje_por_cluster_actual.diff().abs().min(), 2),
      '-> si es menor a ~2 puntos, revisa a mano antes de confiar en el orden.')
print('\nMapa de reetiquetado (etiqueta de esta corrida -> etiqueta Tabla 7):', mapa_reetiquetado)

labels_full = np.array([mapa_reetiquetado[l] for l in labels_full])
print('\nTamanos tras reetiquetar (ya en el orden de la Tabla 7, Cluster 0 a 7):', np.bincount(labels_full))


## 5. Figura S10 — perfil estandarizado (z-score) de los 8 clústeres

In [ ]:
df_perfil = df_filtrado_full.copy()
df_perfil['cluster'] = labels_full

df_perfil['internet_si'] = (df_perfil['FAMI_TIENEINTERNET'] == 'Si').astype(float) * 100
df_perfil['beca_si'] = (df_perfil['ESTU_PAGOMATRICULABECA'] == 'Si').astype(float) * 100
df_perfil['credito_si'] = (df_perfil['ESTU_PAGOMATRICULACREDITO'] == 'Si').astype(float) * 100
df_perfil['autofinanciado_si'] = (df_perfil['ESTU_PAGOMATRICULAPROPIO'] == 'Si').astype(float) * 100

cols_perfil = ['FAMI_ESTRATOVIVIENDA', 'ESTU_HORASSEMANATRABAJA', 'internet_si',
               'beca_si', 'credito_si', 'autofinanciado_si', 'PUNT_GLOBAL']
nombres_col = ['Estrato\n(m\u00e1s alto=mejor)', 'Horas de trabajo\n(m\u00e1s alto=m\u00e1s horas)',
               '% con internet', '% con beca', '% con cr\u00e9dito', '% autofinanciado', 'Puntaje global']

medias_por_cluster = df_perfil.groupby('cluster')[cols_perfil].mean()
tamanos = df_perfil['cluster'].value_counts().sort_index()

z = (medias_por_cluster - medias_por_cluster.mean()) / medias_por_cluster.std()

fig, ax = plt.subplots(figsize=(10, 6.5), dpi=150)
im = ax.imshow(z.values, cmap='RdBu_r', vmin=-1.8, vmax=1.8, aspect='auto')
ax.set_xticks(range(len(nombres_col))); ax.set_xticklabels(nombres_col, rotation=30, ha='right')
etiquetas_y = [f'Cl\u00faster {c} (n={tamanos[c]:,}, {tamanos[c]/tamanos.sum()*100:.1f}%)'
               for c in medias_por_cluster.index]
ax.set_yticks(range(len(etiquetas_y))); ax.set_yticklabels(etiquetas_y)
for i in range(z.shape[0]):
    for j in range(z.shape[1]):
        ax.text(j, i, f'{medias_por_cluster.values[i, j]:.1f}', ha='center', va='center', fontsize=9)
cbar = plt.colorbar(im, ax=ax, shrink=0.85)
cbar.set_label('Z-score')
ax.set_title('Perfil de los 8 cl\u00fasteres (valores medios; color = desviaci\u00f3n respecto al promedio general)\n'
             'Numeraci\u00f3n de cl\u00faster alineada con la Tabla 7 del manuscrito')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'figura_perfiles_8_clusters.png'), dpi=200, bbox_inches='tight', facecolor='white')
plt.show()
medias_por_cluster.to_csv(os.path.join(OUT_DIR, 'medias_por_cluster.csv'))
print("\u2705 Figura S10 guardada en Drive")